# Joint Stress Supervision Ablation

This notebook evaluates **Zonal** models on a **validation-split evaluation** of `disc_dataset_edge_deriv_zonal.h5`.

It compares:
- `Edge_joint` = joint **Stress + LogLife** supervision
- `Edge_no_stress` = **LogLife-only** supervision

Comparable pair families:
- `ArGEnT_self_att_noSDF`
- `PointNetMLPJoint_FP`

`PointNetMLPJoint` has **no life-only checkpoint** under `Zonal/Edge_no_stress` and is reported as missing, then excluded from paired conclusions.


In [ ]:

from __future__ import annotations
import ast, hashlib, importlib.util, inspect, json, re, sys, time, warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

try:
    import h5py
    import torch
except ImportError as exc:
    raise RuntimeError('Install torch and h5py in the selected notebook kernel before executing this comparison.') from exc

CURRENT_DIR = Path.cwd()
REPO_ROOT = CURRENT_DIR if (CURRENT_DIR / 'Uniform').exists() else CURRENT_DIR.parent
if not (REPO_ROOT / 'Uniform').exists():
    raise RuntimeError(f'Repository root not found from {CURRENT_DIR}')
COMPARISON_DIR = REPO_ROOT / 'Comparison'
sys.path.insert(0, str(COMPARISON_DIR))
import eval_helpers as eh

SPLIT_SEED, EVAL_FRACTION = 42, 0.20
COMMIT = __import__('subprocess').check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_ROOT, text=True).strip()

RESULTS_DIR = COMPARISON_DIR / 'results' / '04_joint_stress_supervision'
FIGURES_DIR = RESULTS_DIR / 'figures'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

REGIME = 'Zonal'
DATASET_PATH = REPO_ROOT / 'Data_gen' / 'output' / 'disc_dataset_edge_deriv_zonal.h5'
JOINT_SOURCE = 'Edge'
NO_STRESS_SOURCE = 'Edge_no_stress'
EVALUATION_LABEL = 'validation-split evaluation'

PAIR_TABLE_COLUMNS = [
    'pair_id', 'model_family', 'fp_status', 'stress_variant', 'no_stress_variant',
    'training_config_id', 'split_seed', 'eval_fraction', 'training_fraction',
    'evaluation_subset', 'life_bin', 'n_samples',
    'stress_checkpoint_path', 'no_stress_checkpoint_path',
    'stress_mae_loglife', 'no_stress_mae_loglife', 'delta_mae_loglife',
    'stress_rmse_loglife', 'no_stress_rmse_loglife', 'delta_rmse_loglife',
]

warnings.filterwarnings('ignore', category=FutureWarning)
display(Markdown(f"""**Commit:** `{COMMIT}`

**Results directory:** `{RESULTS_DIR}`

**Evaluation label:** **{EVALUATION_LABEL}**"""))


## Checkpoint discovery and pair-compatibility inputs

The next cell discovers the expected Zonal checkpoints, validates required reconstruction fields, parses lightweight training-script metadata, and writes a checkpoint inventory to `RESULTS_DIR`.


In [ ]:

def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, 'rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def decode(value):
    return value.decode() if isinstance(value, bytes) else value


def to_builtin(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, bytes):
        return value.decode()
    if isinstance(value, dict):
        return {str(k): to_builtin(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_builtin(v) for v in value]
    if hasattr(value, 'detach') and hasattr(value, 'cpu') and hasattr(value, 'tolist'):
        return value.detach().cpu().tolist()
    if hasattr(value, 'tolist') and not isinstance(value, (str, bytes)):
        try:
            return value.tolist()
        except Exception:
            pass
    return value


def missingish(value) -> bool:
    if value is None:
        return True
    try:
        return bool(pd.isna(value))
    except Exception:
        return False


def first_nonmissing(*values):
    for value in values:
        if not missingish(value):
            return value
    return None


def infer_geometry_family(name):
    stem = Path(str(name)).name
    if '_dataset' in stem:
        return stem.split('_dataset', 1)[0]
    parts = stem.split('_')
    return parts[0] if parts else stem


def family_fp_status(family: str) -> str:
    return 'FP' if '_FP' in family else 'non-FP'


def normalized_arch(obj):
    drop = {'out_dim', 'out_channels', 'target_names', 'num_targets'}
    if isinstance(obj, dict):
        return {k: normalized_arch(v) for k, v in sorted(obj.items()) if k not in drop}
    if isinstance(obj, list):
        return [normalized_arch(v) for v in obj]
    return obj


def compact_hash(obj) -> str:
    payload = json.dumps(to_builtin(obj), sort_keys=True, separators=(',', ':')).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()[:16]


def parse_script_metadata(folder: Path) -> dict:
    scripts = sorted(folder.glob('Training_script*.py'), key=lambda p: (p.name != 'Training_script.py', p.name))
    if not scripts:
        return {'training_scripts': []}
    target = scripts[0]
    text = target.read_text(encoding='utf-8')
    tree = ast.parse(text, filename=str(target))
    names = {
        'TARGET_NAMES', 'EXTRA_FEAT_COLS', 'INPUT_COLS', 'QUERY_COLS',
        'H5_FILENAME', 'EXPECTED_REPR', 'Perc_training_data', 'train_data_percent'
    }
    values = {}
    for node in ast.walk(tree):
        if isinstance(node, ast.Assign):
            for t in node.targets:
                if isinstance(t, ast.Name) and t.id in names:
                    try:
                        values[t.id] = ast.literal_eval(node.value)
                    except Exception:
                        pass
        elif isinstance(node, ast.AnnAssign) and isinstance(node.target, ast.Name) and node.target.id in names:
            try:
                values[node.target.id] = ast.literal_eval(node.value)
            except Exception:
                pass
    match = re.search(r'train_test_split\((?P<body>.*?)\)', text, flags=re.S)
    script_eval_fraction, script_split_seed = None, None
    if match:
        body = match.group('body')
        m_frac = re.search(r'test_size\s*=\s*([0-9.]+)', body)
        m_seed = re.search(r'random_state\s*=\s*(\d+)', body)
        if m_frac:
            script_eval_fraction = float(m_frac.group(1))
        if m_seed:
            script_split_seed = int(m_seed.group(1))
    return {
        'training_scripts': [str(p) for p in scripts],
        'script_target_names': to_builtin(values.get('TARGET_NAMES')),
        'script_extra_feat_cols': to_builtin(values.get('EXTRA_FEAT_COLS')),
        'script_input_cols': to_builtin(values.get('INPUT_COLS')),
        'script_query_cols': to_builtin(values.get('QUERY_COLS')),
        'script_h5_filename': values.get('H5_FILENAME'),
        'script_expected_repr': values.get('EXPECTED_REPR'),
        'script_training_fraction': first_nonmissing(values.get('Perc_training_data'), values.get('train_data_percent')),
        'script_eval_fraction': script_eval_fraction,
        'script_split_seed': script_split_seed,
    }


def target_channels_from_payload(payload):
    for key in ('target_mean', 'target_std', 'target_names'):
        if key in payload:
            arr = np.asarray(to_builtin(payload[key]), dtype=object).reshape(-1)
            if arr.size:
                return int(arr.size)
    arch = to_builtin(payload.get('arch') or {})
    for key in ('out_dim', 'out_channels'):
        if key in arch:
            return int(arch[key])
    return None


def discover() -> pd.DataFrame:
    rows = []
    required = ['arch', 'model_state', 'target_mean', 'target_std', 'coord_center', 'coord_half_range']
    for ablation_source in [JOINT_SOURCE, NO_STRESS_SOURCE]:
        source_dir = REPO_ROOT / REGIME / ablation_source
        if not source_dir.exists():
            continue
        family_dirs = [p for p in sorted(source_dir.iterdir()) if p.is_dir() and (p / 'Trained_models').exists()]
        for folder in family_dirs:
            family = folder.name
            script_meta = parse_script_metadata(folder)
            checkpoints = sorted((folder / 'Trained_models').glob('*.pt'))
            for path in checkpoints:
                record = {
                    'regime': REGIME,
                    'ablation_source': ablation_source,
                    'ablation': ablation_source,
                    'model_family': family,
                    'variant_name': path.stem,
                    'fp_status': family_fp_status(family),
                    'checkpoint_path': str(path),
                    'file_size_bytes': int(path.stat().st_size),
                    'sha256': sha256(path),
                    **script_meta,
                }
                try:
                    payload = torch.load(path, map_location='cpu', weights_only=False)
                    missing = [k for k in required if k not in payload]
                    arch = to_builtin(payload.get('arch') or {})
                    target_channels = target_channels_from_payload(payload)
                    stress_mode = 'with_stress' if target_channels == 2 else 'no_stress' if target_channels == 1 else 'unknown'
                    feature_signature = {
                        'extra_feat_cols': to_builtin(payload.get('extra_feat_cols')) or to_builtin(script_meta.get('script_extra_feat_cols')) or [],
                        'input_cols': to_builtin(script_meta.get('script_input_cols')) or to_builtin(arch.get('input_cols')) or [0, 1],
                        'query_cols': to_builtin(script_meta.get('script_query_cols')) or [0, 1],
                    }
                    norm_arch = normalized_arch(arch)
                    record.update({
                        'loaded_successfully': not missing,
                        'status': 'discovered' if not missing else 'incompatible: missing ' + ', '.join(missing),
                        'missing_keys': missing,
                        'arch': arch,
                        'normalized_arch_hash': compact_hash(norm_arch),
                        'arch_hash': compact_hash(arch),
                        'target_names': to_builtin(payload.get('target_names')),
                        'target_channels': target_channels,
                        'stress_target_mode': stress_mode,
                        'extra_feat_cols': to_builtin(payload.get('extra_feat_cols')),
                        'representation': decode(payload.get('representation')) if payload.get('representation') is not None else None,
                        'h5_filename': payload.get('h5_filename') or (Path(payload['h5_path']).name if payload.get('h5_path') else None),
                        'geometry_family': payload.get('geometry_family'),
                        'train_sample_ids': to_builtin(payload.get('train_sample_ids')),
                        'val_sample_ids': to_builtin(payload.get('val_sample_ids')),
                        'split_seed': to_builtin(payload.get('split_seed')),
                        'eval_fraction': to_builtin(payload.get('eval_fraction')),
                        'training_fraction': first_nonmissing(payload.get('train_data_percent'), payload.get('Perc_training_data')),
                        'feature_signature': feature_signature,
                    })
                    record['effective_h5_filename'] = first_nonmissing(record.get('h5_filename'), record.get('script_h5_filename'))
                    record['effective_representation'] = first_nonmissing(record.get('representation'), record.get('script_expected_repr'))
                    record['effective_geometry_family'] = first_nonmissing(record.get('geometry_family'), infer_geometry_family(record.get('effective_h5_filename') or DATASET_PATH.name))
                    record['effective_split_seed'] = first_nonmissing(record.get('split_seed'), record.get('script_split_seed'))
                    record['effective_eval_fraction'] = first_nonmissing(record.get('eval_fraction'), record.get('script_eval_fraction'))
                    record['effective_training_fraction'] = first_nonmissing(record.get('training_fraction'), record.get('script_training_fraction'))
                    record['training_config_id'] = compact_hash({
                        'family': family,
                        'fp_status': record['fp_status'],
                        'arch': norm_arch,
                        'feature_signature': feature_signature,
                        'dataset': record['effective_h5_filename'],
                        'representation': record['effective_representation'],
                    })
                except Exception as exc:
                    record.update({
                        'loaded_successfully': False,
                        'status': 'incompatible: ' + repr(exc),
                    })
                rows.append(record)
    report = pd.DataFrame(rows).sort_values(['ablation_source', 'model_family', 'checkpoint_path']).reset_index(drop=True)
    inventory_cols = [
        'checkpoint_path', 'model_family', 'variant_name', 'fp_status', 'stress_target_mode',
        'training_config_id', 'loaded_successfully', 'status', 'effective_h5_filename',
        'effective_representation', 'effective_split_seed', 'effective_eval_fraction',
    ]
    inventory = report[inventory_cols].copy() if not report.empty else pd.DataFrame(columns=inventory_cols)
    eh.save_table(inventory, RESULTS_DIR, 'checkpoint_inventory')
    eh.save_json(report.to_dict(orient='records'), RESULTS_DIR, 'checkpoint_inventory')
    return report


checkpoint_report = discover()
display(checkpoint_report[[
    'checkpoint_path', 'model_family', 'variant_name', 'fp_status',
    'stress_target_mode', 'training_config_id', 'loaded_successfully', 'status'
]] if not checkpoint_report.empty else pd.DataFrame(columns=[
    'checkpoint_path', 'model_family', 'variant_name', 'fp_status',
    'stress_target_mode', 'training_config_id', 'loaded_successfully', 'status'
]))


## HDF5 loading, deterministic evaluation split, and fairness checks

This notebook uses one shared Zonal edge dataset and one deterministic 80/20 geometry split for every evaluated model. Fair paired conclusions are restricted to model families that pass the explicit checks below.


In [ ]:

def load_samples(path: Path):
    samples = []
    with h5py.File(path, 'r') as h5:
        representation = decode(h5.attrs.get('representation', ''))
        if representation != 'edge':
            raise ValueError(f'{path.name}: expected representation edge, got {representation!r}')
        for key in sorted(h5['samples'].keys()):
            g = h5['samples'][key]
            def arr(name, default=None):
                return np.asarray(g[name]) if name in g else default
            coords = arr('node_coords_mm')
            stress = arr('stress_max_vm')
            life = arr('life_raw')
            if coords is None or stress is None or life is None:
                raise ValueError(f'{path.name}/{key}: missing required target fields')
            sample_id = decode(g.attrs.get('sample_id', key))
            attrs = {str(k): decode(v) for k, v in g.attrs.items()}
            samples.append({
                'sample_key': key,
                'sample_id': str(sample_id),
                'attrs': attrs,
                'coords': coords.astype('float32'),
                'stress': stress.astype('float32').reshape(-1),
                'loglife': np.log10(np.clip(life.astype('float64').reshape(-1), 1e-30, None)).astype('float32'),
                'zone_id': arr('zone_id', np.full(len(coords), -1)).reshape(-1),
                'subzone_id': arr('subzone_id', np.full(len(coords), np.nan)).reshape(-1),
                'arc_length_mm': arr('arc_length_mm', np.arange(len(coords), dtype='float32')).reshape(-1),
                'node_features': arr('node_features', np.empty((len(coords), 0), dtype='float32')),
            })
    return samples


def split_samples(samples):
    rng = np.random.default_rng(SPLIT_SEED)
    order = rng.permutation(len(samples))
    n_eval = max(1, int(round(len(samples) * EVAL_FRACTION)))
    eval_pos = np.sort(order[:n_eval]).tolist()
    train_pos = np.sort(order[n_eval:]).tolist()
    return train_pos, eval_pos


def compare_required(left, right, field, label, reasons):
    lv = left.get(field)
    rv = right.get(field)
    if missingish(lv) or missingish(rv):
        reasons.append(f'{label} unavailable on one or both checkpoints')
        return False
    if isinstance(lv, float) or isinstance(rv, float):
        if abs(float(lv) - float(rv)) > 1e-12:
            reasons.append(f'{label} mismatch ({lv!r} vs {rv!r})')
            return False
        return True
    if lv != rv:
        reasons.append(f'{label} mismatch ({lv!r} vs {rv!r})')
        return False
    return True


def compare_optional(left, right, field, label, reasons):
    lv = left.get(field)
    rv = right.get(field)
    if missingish(lv) and missingish(rv):
        return True
    if missingish(lv) or missingish(rv):
        reasons.append(f'{label} unavailable on one checkpoint')
        return False
    if isinstance(lv, float) or isinstance(rv, float):
        if abs(float(lv) - float(rv)) > 1e-12:
            reasons.append(f'{label} mismatch ({lv!r} vs {rv!r})')
            return False
        return True
    if lv != rv:
        reasons.append(f'{label} mismatch ({lv!r} vs {rv!r})')
        return False
    return True


def build_valid_pairs(report: pd.DataFrame):
    valid_rows, excluded_rows = [], []
    if report.empty:
        return pd.DataFrame(), pd.DataFrame(), 'No checkpoints were discovered.'
    discovered = report[report['status'] == 'discovered'].copy()
    stress = discovered[discovered['stress_target_mode'] == 'with_stress'].copy()
    no_stress = discovered[discovered['stress_target_mode'] == 'no_stress'].copy()
    for _, stress_row in stress.iterrows():
        same_family = no_stress[
            (no_stress['model_family'] == stress_row['model_family']) &
            (no_stress['fp_status'] == stress_row['fp_status'])
        ]
        if same_family.empty:
            excluded_rows.append({
                'model_family': stress_row['model_family'],
                'fp_status': stress_row['fp_status'],
                'stress_checkpoint_path': stress_row['checkpoint_path'],
                'no_stress_checkpoint_path': None,
                'reason': 'no no-stress counterpart discovered for same model family + FP status',
            })
            continue
        matches = []
        for _, ns_row in same_family.iterrows():
            reasons = []
            compare_required(stress_row, ns_row, 'effective_h5_filename', 'dataset filename', reasons)
            compare_required(stress_row, ns_row, 'effective_representation', 'representation', reasons)
            compare_required(stress_row, ns_row, 'effective_geometry_family', 'geometry family', reasons)
            compare_required(stress_row, ns_row, 'normalized_arch_hash', 'architecture/configuration identifier', reasons)
            compare_required(stress_row, ns_row, 'training_config_id', 'training configuration identifier', reasons)
            compare_required(stress_row, ns_row, 'feature_signature', 'feature signature', reasons)
            compare_required(stress_row, ns_row, 'effective_split_seed', 'split seed', reasons)
            compare_required(stress_row, ns_row, 'effective_eval_fraction', 'evaluation fraction', reasons)
            compare_optional(stress_row, ns_row, 'effective_training_fraction', 'training fraction', reasons)
            if reasons:
                excluded_rows.append({
                    'model_family': stress_row['model_family'],
                    'fp_status': stress_row['fp_status'],
                    'stress_checkpoint_path': stress_row['checkpoint_path'],
                    'no_stress_checkpoint_path': ns_row['checkpoint_path'],
                    'reason': '; '.join(reasons),
                })
            else:
                matches.append(ns_row)
        if len(matches) == 1:
            ns_row = matches[0]
            valid_rows.append({
                'pair_id': f"{stress_row['model_family']}__{stress_row['training_config_id']}",
                'model_family': stress_row['model_family'],
                'fp_status': stress_row['fp_status'],
                'stress_variant': stress_row['variant_name'],
                'no_stress_variant': ns_row['variant_name'],
                'stress_checkpoint_path': stress_row['checkpoint_path'],
                'no_stress_checkpoint_path': ns_row['checkpoint_path'],
                'training_config_id': stress_row['training_config_id'],
                'split_seed': int(stress_row['effective_split_seed']) if not missingish(stress_row['effective_split_seed']) else np.nan,
                'eval_fraction': float(stress_row['effective_eval_fraction']) if not missingish(stress_row['effective_eval_fraction']) else np.nan,
                'training_fraction': float(stress_row['effective_training_fraction']) if not missingish(stress_row['effective_training_fraction']) else np.nan,
                'effective_h5_filename': stress_row['effective_h5_filename'],
                'effective_representation': stress_row['effective_representation'],
                'effective_geometry_family': stress_row['effective_geometry_family'],
            })
        elif len(matches) > 1:
            excluded_rows.append({
                'model_family': stress_row['model_family'],
                'fp_status': stress_row['fp_status'],
                'stress_checkpoint_path': stress_row['checkpoint_path'],
                'no_stress_checkpoint_path': None,
                'reason': 'multiple metadata-compatible no-stress counterparts discovered; pairing is ambiguous',
            })
    valid_pairs = pd.DataFrame(valid_rows)
    excluded = pd.DataFrame(excluded_rows)
    eh.save_table(valid_pairs if not valid_pairs.empty else pd.DataFrame(columns=[
        'pair_id', 'model_family', 'fp_status', 'stress_variant', 'no_stress_variant',
        'stress_checkpoint_path', 'no_stress_checkpoint_path', 'training_config_id',
        'split_seed', 'eval_fraction', 'training_fraction', 'effective_h5_filename',
        'effective_representation', 'effective_geometry_family'
    ]), RESULTS_DIR, 'stress_vs_no_stress_pair_inventory')
    eh.save_table(excluded if not excluded.empty else pd.DataFrame(columns=[
        'model_family', 'fp_status', 'stress_checkpoint_path', 'no_stress_checkpoint_path', 'reason'
    ]), RESULTS_DIR, 'stress_vs_no_stress_excluded_candidates')

    stress_found = stress[['model_family', 'fp_status', 'variant_name', 'checkpoint_path']].to_dict(orient='records') if not stress.empty else []
    no_stress_found = no_stress[['model_family', 'fp_status', 'variant_name', 'checkpoint_path']].to_dict(orient='records') if not no_stress.empty else []
    excluded_found = excluded.to_dict(orient='records') if not excluded.empty else []
    diagnostic_text = f"""No valid stress/no-stress comparison pairs found.

Discovered stress variants:
{json.dumps(stress_found, indent=2)}

Discovered no-stress variants:
{json.dumps(no_stress_found, indent=2)}

Excluded candidates and reasons:
{json.dumps(excluded_found, indent=2)}

Expected valid pairing rule:
same model family + same FP status + same split/seed/configuration,
differing only in stress condition."""
    return valid_pairs, excluded, diagnostic_text


dataset_available = DATASET_PATH.exists()
dataset_status = {
    'dataset_path': str(DATASET_PATH),
    'dataset_available': bool(dataset_available),
    'timestamp_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
}
if dataset_available:
    samples = load_samples(DATASET_PATH)
    train_pos, eval_pos = split_samples(samples)
else:
    samples, train_pos, eval_pos = [], [], []
    dataset_status['reason'] = 'Dataset file is unavailable in this checkout.'
eh.save_json(dataset_status, RESULTS_DIR, 'dataset_status')

split_record = {
    'regime': REGIME,
    'hdf5_filename': DATASET_PATH.name,
    'geometry_family': infer_geometry_family(DATASET_PATH.name),
    'total_geometry_count': len(samples),
    'evaluation_geometry_count': len(eval_pos),
    'split_seed': SPLIT_SEED,
    'split_fraction': EVAL_FRACTION,
    'training_sample_ids': [samples[i]['sample_id'] for i in train_pos],
    'evaluation_sample_ids': [samples[i]['sample_id'] for i in eval_pos],
    'training_positional_indices': train_pos,
    'evaluation_positional_indices': eval_pos,
    'timestamp_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    'notebook_commit_sha': COMMIT,
    'evaluation_label': EVALUATION_LABEL,
    'dataset_available': bool(dataset_available),
    'independence_basis': 'A deterministic geometry holdout is used, but checkpoint-selection independence is not proved.',
}
eh.save_json(split_record, RESULTS_DIR, 'evaluation_split_provenance')

valid_pairs, excluded_candidates, no_pair_diagnostic = build_valid_pairs(checkpoint_report)
if valid_pairs.empty:
    (RESULTS_DIR / 'stress_vs_no_stress_diagnostic.md').write_text(no_pair_diagnostic, encoding='utf-8')
else:
    diagnostic_text = valid_pairs[[
        'model_family', 'fp_status', 'stress_variant', 'no_stress_variant',
        'training_config_id', 'stress_checkpoint_path', 'no_stress_checkpoint_path'
    ]].to_markdown(index=False)
    (RESULTS_DIR / 'stress_vs_no_stress_diagnostic.md').write_text(diagnostic_text, encoding='utf-8')

eh.save_json({
    'valid_pairs': valid_pairs.to_dict(orient='records'),
    'excluded_candidates': excluded_candidates.to_dict(orient='records') if not excluded_candidates.empty else [],
    'dataset_available': bool(dataset_available),
}, RESULTS_DIR, 'stress_vs_no_stress_diagnostic')

display(pd.DataFrame([{
    'regime': REGIME,
    'geometries': len(samples),
    'evaluation_geometries': len(eval_pos),
    'dataset_available': bool(dataset_available),
    'label': EVALUATION_LABEL,
}]))
display(valid_pairs if not valid_pairs.empty else pd.DataFrame(columns=[
    'model_family', 'fp_status', 'stress_variant', 'no_stress_variant', 'training_config_id'
]))
display(excluded_candidates if not excluded_candidates.empty else pd.DataFrame(columns=['reason']))


## Model reconstruction and shared-geometry inference

The next cell reconstructs every discovered Zonal checkpoint, evaluates the **same validation-split geometries** for every model, and writes node-level predictions to `RESULTS_DIR`.


In [ ]:

def import_local(path, name):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


def reconstruct(row):
    folder = Path(row['checkpoint_path']).parent.parent
    family = row['model_family']
    ckpt = torch.load(row['checkpoint_path'], map_location='cpu', weights_only=False)
    arch = dict(ckpt['arch'])
    pn = import_local(folder / 'pn_models.py', f'pn_{folder.parent.name}_{folder.name}_{family}_{row["ablation_source"]}')
    sys.modules['pn_models'] = pn
    if family == 'PointNetMLPJoint_FP':
        if not hasattr(pn, 'build_fp_model_from_arch'):
            raise RuntimeError('FP checkpoint requires build_fp_model_from_arch; refusing regular PointNet fallback')
        model = pn.build_fp_model_from_arch(arch)
    elif family in ('PointNetMLPJoint', 'PointNetMLPJoint_weighted'):
        model = pn.build_model_from_arch(arch)
    else:
        bench = import_local(folder / 'benchmarks.py', f'bench_{row["ablation_source"]}_{family}')
        if not hasattr(bench, 'ArGEnTDeepONet'):
            raise RuntimeError('No ArGEnTDeepONet found in own benchmarks.py')
        defaults = {
            'hidden_dim': 128, 'num_heads': 4, 'num_layers': 2, 'output_dim': 128,
            'out_channels': 1, 'attention_type': 'self', 'use_sdf': False, 'in_ch_geom': 2,
        }
        defaults.update({k: v for k, v in arch.items() if k in defaults})
        if 'out_channels' not in arch and 'bias' in ckpt['model_state']:
            defaults['out_channels'] = int(ckpt['model_state']['bias'].shape[0])
        model = bench.ArGEnTDeepONet(**defaults)
    model.load_state_dict(ckpt['model_state'], strict=True)
    model.eval()
    return model, ckpt


def source_column_matrix(sample):
    node_features = sample['node_features']
    if node_features.ndim == 1:
        node_features = node_features[:, None]
    cols = [sample['coords'][:, 0], sample['coords'][:, 1], sample['zone_id'], sample['arc_length_mm']]
    for j in range(node_features.shape[1]):
        cols.append(node_features[:, j])
    return np.column_stack(cols).astype('float32')


def feature_matrix(sample, ckpt):
    extra_feat_cols = [int(c) for c in (ckpt.get('extra_feat_cols', []) or [])]
    source = source_column_matrix(sample)
    center = np.asarray(ckpt['coord_center'], dtype='float32')
    half = np.asarray(ckpt['coord_half_range'], dtype='float32')
    coords = (sample['coords'] - center) / np.maximum(half, 1e-8)
    if extra_feat_cols:
        if source.shape[1] <= max(extra_feat_cols):
            raise ValueError(f'source tensor width {source.shape[1]} too small for extra_feat_cols={extra_feat_cols}')
        extra = source[:, extra_feat_cols].astype('float32')
        stats = ckpt.get('extra_feat_stats')
        if stats is None:
            raise ValueError('missing extra-feature normalization statistics')
        means, stds = [], []
        for col in extra_feat_cols:
            entry = stats.get(col, stats.get(str(col))) if isinstance(stats, dict) else None
            if entry is None:
                raise ValueError(f'missing extra_feat_stats entry for column {col}')
            if isinstance(entry, dict):
                means.append(float(entry['mean']))
                stds.append(float(entry['std']))
            else:
                means.append(float(entry[0]))
                stds.append(float(entry[1]))
        extra = (extra - np.asarray(means, dtype='float32')) / np.maximum(np.asarray(stds, dtype='float32'), 1e-8)
    else:
        extra = np.empty((len(sample['coords']), 0), dtype='float32')
    return coords.astype('float32'), extra.astype('float32')


def predict(model, sample, ckpt):
    coords, extra = feature_matrix(sample, ckpt)
    x = torch.from_numpy(coords[None])
    q = x.clone()
    kwargs = {}
    sig = inspect.signature(model.forward)
    if 'geom_feats' in sig.parameters and extra.shape[1] > 0:
        kwargs['geom_feats'] = torch.from_numpy(extra[None])
    if 'mask' in sig.parameters:
        kwargs['mask'] = torch.ones((1, q.shape[1]), dtype=torch.bool)
    if 'kv_mask' in sig.parameters:
        kwargs['kv_mask'] = torch.ones((1, x.shape[1]), dtype=torch.bool)
    with torch.no_grad():
        out = model(x, q, **kwargs)
    out = out.detach().cpu().numpy()
    if out.ndim != 3 or out.shape[0] != 1:
        raise ValueError(f'prediction shape unexpected: {out.shape}')
    mean = np.asarray(ckpt['target_mean'])
    std = np.asarray(ckpt['target_std'])
    out = out * std + mean
    if out.shape[2] == 2:
        return out[0, :, 0].astype('float32'), out[0, :, 1].astype('float32')
    if out.shape[2] == 1:
        return np.full(out.shape[1], np.nan, dtype='float32'), out[0, :, 0].astype('float32')
    raise ValueError(f'unexpected output channels: {out.shape[2]}')


node_columns = [
    'regime', 'ablation', 'ablation_source', 'model_family', 'fp_status', 'stress_target_mode',
    'variant_name', 'checkpoint_path', 'training_config_id', 'evaluation_label', 'sample_key',
    'sample_id', 'node_idx', 'x_mm', 'r_mm', 'zone_id', 'subzone_id', 'arc_length_mm',
    'true_stress', 'pred_stress', 'true_loglife', 'pred_loglife', 'zone_name', 'subzone_name'
]

node_frames, load_errors = [], []
discovered = checkpoint_report[checkpoint_report['status'] == 'discovered'].copy()
for _, row in discovered.iterrows():
    if not dataset_available:
        break
    try:
        model, ckpt = reconstruct(row)
        predicted_ids = []
        for i in split_record['evaluation_positional_indices']:
            s = samples[i]
            pred_stress, pred_loglife = predict(model, s, ckpt)
            predicted_ids.append(s['sample_id'])
            base = pd.DataFrame({
                'regime': row['regime'],
                'ablation': row['ablation'],
                'ablation_source': row['ablation_source'],
                'model_family': row['model_family'],
                'fp_status': row['fp_status'],
                'stress_target_mode': row['stress_target_mode'],
                'variant_name': row['variant_name'],
                'checkpoint_path': row['checkpoint_path'],
                'training_config_id': row['training_config_id'],
                'evaluation_label': EVALUATION_LABEL,
                'sample_key': s['sample_key'],
                'sample_id': s['sample_id'],
                'node_idx': np.arange(len(s['coords'])),
                'x_mm': s['coords'][:, 0],
                'r_mm': s['coords'][:, 1],
                'zone_id': s['zone_id'],
                'subzone_id': s['subzone_id'],
                'arc_length_mm': s['arc_length_mm'],
                'true_stress': s['stress'],
                'pred_stress': pred_stress,
                'true_loglife': s['loglife'],
                'pred_loglife': pred_loglife,
            })
            base['zone_name'] = base.zone_id.map(eh.ZONE_ID_TO_NAME)
            base['subzone_name'] = base.subzone_id.map(eh.SUBZONE_ID_TO_NAME)
            node_frames.append(base)
        load_errors.append({
            'checkpoint_path': row['checkpoint_path'],
            'model_family': row['model_family'],
            'status': 'ok',
            'prediction_sample_ids_match_eval_split': predicted_ids == split_record['evaluation_sample_ids'],
            'n_evaluation_geometries': len(predicted_ids),
        })
    except Exception as exc:
        load_errors.append({
            'checkpoint_path': row['checkpoint_path'],
            'model_family': row['model_family'],
            'status': 'load/inference failed: ' + repr(exc),
            'prediction_sample_ids_match_eval_split': False,
            'n_evaluation_geometries': 0,
        })

if not dataset_available:
    load_errors.append({
        'checkpoint_path': None,
        'model_family': None,
        'status': f'dataset unavailable: {DATASET_PATH}',
        'prediction_sample_ids_match_eval_split': False,
        'n_evaluation_geometries': 0,
    })

node_results = pd.concat(node_frames, ignore_index=True) if node_frames else pd.DataFrame(columns=node_columns)
inference_errors = pd.DataFrame(load_errors)
eh.save_table(inference_errors, RESULTS_DIR, 'evaluation_coverage')
eh.save_json(load_errors, RESULTS_DIR, 'inference_errors')
display(inference_errors)


## Metrics and paired joint-vs-life-only deltas

Positive paired improvements below mean the **joint** model achieved a **lower absolute LogLife error** than its life-only counterpart.


In [ ]:

def loglife_subset_metrics(frame: pd.DataFrame, group_cols) -> pd.DataFrame:
    if frame.empty:
        return pd.DataFrame(columns=group_cols + ['life_bin', 'n_samples', 'mae_loglife', 'rmse_loglife'])
    rows = []
    for keys, g in frame.groupby(group_cols):
        if not isinstance(keys, tuple):
            keys = (keys,)
        base = dict(zip(group_cols, keys))
        t_all = g['true_loglife'].to_numpy()
        p_all = g['pred_loglife'].to_numpy()
        for life_bin, lo, hi in eh.ALL_LIFE_BIN_DEFS:
            mask = np.ones_like(t_all, dtype=bool)
            if lo is not None:
                mask &= (t_all >= lo)
            if hi is not None:
                mask &= (t_all < hi)
            n = int(mask.sum())
            err = p_all[mask] - t_all[mask]
            rows.append({
                **base,
                'life_bin': life_bin,
                'n_samples': n,
                'mae_loglife': float(np.mean(np.abs(err))) if n else np.nan,
                'rmse_loglife': float(np.sqrt(np.mean(err ** 2))) if n else np.nan,
                'unstable': bool(n < eh.MIN_BIN_NODES),
            })
    return pd.DataFrame(rows)


def stress_metrics_by_checkpoint(frame: pd.DataFrame) -> pd.DataFrame:
    if frame.empty:
        return pd.DataFrame(columns=['checkpoint_path', 'stress_mae', 'stress_rmse'])
    rows = []
    for checkpoint_path, g in frame.groupby('checkpoint_path'):
        finite = np.isfinite(g['pred_stress'].to_numpy()) & np.isfinite(g['true_stress'].to_numpy())
        rows.append({
            'checkpoint_path': checkpoint_path,
            'stress_mae': float(np.mean(np.abs(g.loc[finite, 'pred_stress'] - g.loc[finite, 'true_stress']))) if finite.any() else np.nan,
            'stress_rmse': float(np.sqrt(np.mean((g.loc[finite, 'pred_stress'] - g.loc[finite, 'true_stress']) ** 2))) if finite.any() else np.nan,
        })
    return pd.DataFrame(rows)


group_cols = ['regime', 'ablation', 'ablation_source', 'model_family', 'fp_status', 'stress_target_mode', 'checkpoint_path', 'training_config_id']
life_band_metrics = loglife_subset_metrics(node_results, group_cols)
grouped_regions = eh.grouped_region_metrics_from_nodes(node_results) if not node_results.empty else pd.DataFrame()
geom = eh.geometry_level_metrics(node_results) if not node_results.empty else pd.DataFrame()
stress_metrics = stress_metrics_by_checkpoint(node_results)

summary_rows = []
discovered = checkpoint_report[checkpoint_report['status'] == 'discovered'].copy()
for _, row in discovered.iterrows():
    sub = life_band_metrics[
        (life_band_metrics['checkpoint_path'] == row['checkpoint_path']) &
        (life_band_metrics['life_bin'] == eh.FULL_TEST_SET_LABEL)
    ]
    stress_sub = stress_metrics[stress_metrics['checkpoint_path'] == row['checkpoint_path']]
    summary_rows.append({
        'evaluation_label': EVALUATION_LABEL,
        'regime': REGIME,
        'ablation': row['ablation_source'],
        'model_family': row['model_family'],
        'fp_status': row['fp_status'],
        'stress_target_mode': row['stress_target_mode'],
        'variant_name': row['variant_name'],
        'checkpoint_path': row['checkpoint_path'],
        'training_config_id': row['training_config_id'],
        'n_samples_full_test': int(sub.iloc[0]['n_samples']) if not sub.empty else 0,
        'LogLife_MAE': float(sub.iloc[0]['mae_loglife']) if not sub.empty else np.nan,
        'LogLife_RMSE': float(sub.iloc[0]['rmse_loglife']) if not sub.empty else np.nan,
        'Stress_MAE': float(stress_sub.iloc[0]['stress_mae']) if not stress_sub.empty else np.nan,
        'Stress_RMSE': float(stress_sub.iloc[0]['stress_rmse']) if not stress_sub.empty else np.nan,
    })
summary_table = pd.DataFrame(summary_rows)

pair_rows = []
for _, pair in valid_pairs.iterrows():
    stress_sub = life_band_metrics[life_band_metrics['checkpoint_path'] == pair['stress_checkpoint_path']].copy()
    no_stress_sub = life_band_metrics[life_band_metrics['checkpoint_path'] == pair['no_stress_checkpoint_path']].copy()
    if stress_sub.empty or no_stress_sub.empty:
        continue
    merged = stress_sub.merge(no_stress_sub, on='life_bin', suffixes=('_stress', '_no_stress'))
    for _, row in merged.iterrows():
        pair_rows.append({
            'pair_id': pair['pair_id'],
            'model_family': pair['model_family'],
            'fp_status': pair['fp_status'],
            'stress_variant': pair['stress_variant'],
            'no_stress_variant': pair['no_stress_variant'],
            'training_config_id': pair['training_config_id'],
            'split_seed': pair['split_seed'],
            'eval_fraction': pair['eval_fraction'],
            'training_fraction': pair['training_fraction'],
            'evaluation_subset': EVALUATION_LABEL,
            'life_bin': row['life_bin'],
            'n_samples': int(row['n_samples_stress']) if not pd.isna(row['n_samples_stress']) else int(row['n_samples_no_stress']),
            'stress_checkpoint_path': pair['stress_checkpoint_path'],
            'no_stress_checkpoint_path': pair['no_stress_checkpoint_path'],
            'stress_mae_loglife': float(row['mae_loglife_stress']) if not pd.isna(row['mae_loglife_stress']) else np.nan,
            'no_stress_mae_loglife': float(row['mae_loglife_no_stress']) if not pd.isna(row['mae_loglife_no_stress']) else np.nan,
            'delta_mae_loglife': float(row['mae_loglife_no_stress'] - row['mae_loglife_stress']) if not (pd.isna(row['mae_loglife_no_stress']) or pd.isna(row['mae_loglife_stress'])) else np.nan,
            'stress_rmse_loglife': float(row['rmse_loglife_stress']) if not pd.isna(row['rmse_loglife_stress']) else np.nan,
            'no_stress_rmse_loglife': float(row['rmse_loglife_no_stress']) if not pd.isna(row['rmse_loglife_no_stress']) else np.nan,
            'delta_rmse_loglife': float(row['rmse_loglife_no_stress'] - row['rmse_loglife_stress']) if not (pd.isna(row['rmse_loglife_no_stress']) or pd.isna(row['rmse_loglife_stress'])) else np.nan,
        })
paired_table = pd.DataFrame(pair_rows, columns=PAIR_TABLE_COLUMNS)
if paired_table.empty:
    paired_table = pd.DataFrame(columns=PAIR_TABLE_COLUMNS)

paired_summary = pd.DataFrame()
if not paired_table.empty:
    paired_summary = paired_table.groupby(['model_family', 'fp_status', 'life_bin'], as_index=False).agg(
        n_rows=('pair_id', 'size'),
        n_samples=('n_samples', 'sum'),
        median_delta_mae_loglife=('delta_mae_loglife', 'median'),
        median_delta_rmse_loglife=('delta_rmse_loglife', 'median'),
        fraction_stress_better_mae=('delta_mae_loglife', lambda s: float((s > 0).mean())),
        fraction_stress_better_rmse=('delta_rmse_loglife', lambda s: float((s > 0).mean())),
    )

if summary_table.empty:
    summary_table = pd.DataFrame(columns=[
        'evaluation_label', 'regime', 'ablation', 'model_family', 'fp_status', 'stress_target_mode',
        'variant_name', 'checkpoint_path', 'training_config_id', 'n_samples_full_test',
        'LogLife_MAE', 'LogLife_RMSE', 'Stress_MAE', 'Stress_RMSE'
    ])
if life_band_metrics.empty:
    life_band_metrics = pd.DataFrame(columns=group_cols + ['life_bin', 'n_samples', 'mae_loglife', 'rmse_loglife', 'unstable'])
if grouped_regions.empty:
    grouped_regions = pd.DataFrame(columns=['regime', 'ablation', 'model_family', 'grouped_region', 'n_nodes', 'LogLife_MAE', 'LogLife_RMSE', 'Stress_MAE', 'Stress_RMSE', 'status'])

if valid_pairs.empty:
    (RESULTS_DIR / 'stress_vs_no_stress_diagnostic.md').write_text(no_pair_diagnostic, encoding='utf-8')

eh.save_table(summary_table, RESULTS_DIR, 'summary_table')
eh.save_table(life_band_metrics, RESULTS_DIR, 'life_band_metrics')
eh.save_table(grouped_regions, RESULTS_DIR, 'grouped_region_metrics')
eh.save_table(paired_table, RESULTS_DIR, 'stress_vs_no_stress_paired_metrics')
eh.save_table(paired_summary if not paired_summary.empty else pd.DataFrame(columns=[
    'model_family', 'fp_status', 'life_bin', 'n_rows', 'n_samples',
    'median_delta_mae_loglife', 'median_delta_rmse_loglife',
    'fraction_stress_better_mae', 'fraction_stress_better_rmse'
]), RESULTS_DIR, 'paired_summary')

run_metadata = {
    'commit_sha': COMMIT,
    'evaluation_label': EVALUATION_LABEL,
    'results_dir': str(RESULTS_DIR),
    'dataset_available': bool(dataset_available),
    'valid_pair_ids': valid_pairs['pair_id'].tolist() if not valid_pairs.empty else [],
    'comparison_delta_convention': 'delta = no_stress - stress; positive means the stress-supervised model has lower error.',
    'nonexistent_regular_pointnet_no_stress_model_included': False,
}
eh.save_json(run_metadata, RESULTS_DIR, 'run_metadata')

display(summary_table)
display(paired_table)


## Figures

Each figure is saved to `RESULTS_DIR/figures` (PNG, with PDF fallback where available).


In [ ]:

def save_fig(fig, name):
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)
    png = FIGURES_DIR / f'{name}.png'
    pdf = FIGURES_DIR / f'{name}.pdf'
    fig.savefig(png, dpi=150, bbox_inches='tight')
    fig.savefig(pdf, bbox_inches='tight')
    plt.close(fig)
    return png, pdf


figure_inventory = []
life_bin_order = [label for label, _, _ in eh.ALL_LIFE_BIN_DEFS]

for pair_id, g in paired_table.groupby('pair_id') if not paired_table.empty else []:
    d = g.set_index('life_bin').reindex(life_bin_order).reset_index()
    model_family = d['model_family'].dropna().iloc[0]
    fp_status = d['fp_status'].dropna().iloc[0]
    stress_variant = d['stress_variant'].dropna().iloc[0]
    no_stress_variant = d['no_stress_variant'].dropna().iloc[0]
    x = np.arange(len(d), dtype=float)
    tick_labels = [f"{lb}{chr(10)}(n={int(n)})" if not pd.isna(n) else lb for lb, n in zip(d['life_bin'], d['n_samples'])]
    fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharex='col')
    for ax, stress_col, no_stress_col, metric in [
        (axes[0, 0], 'stress_mae_loglife', 'no_stress_mae_loglife', 'MAE'),
        (axes[0, 1], 'stress_rmse_loglife', 'no_stress_rmse_loglife', 'RMSE'),
    ]:
        ax.plot(x, d[stress_col], marker='o', label=f'{stress_variant} (with stress)')
        ax.plot(x, d[no_stress_col], marker='s', label=f'{no_stress_variant} (no stress)')
        for xi, ys, yn in zip(x, d[stress_col], d[no_stress_col]):
            if pd.notna(ys) and pd.notna(yn):
                ax.plot([xi, xi], [ys, yn], color='0.7', lw=1)
        ax.set_ylabel(f'{metric}(log-life) [decades]')
        ax.set_title(f'{metric} by life bin')
        ax.grid(axis='y', alpha=0.3)
    axes[0, 0].legend(fontsize=8)
    for ax, delta_col, metric in [
        (axes[1, 0], 'delta_mae_loglife', 'ΔMAE = no-stress − stress'),
        (axes[1, 1], 'delta_rmse_loglife', 'ΔRMSE = no-stress − stress'),
    ]:
        colors = ['#d62728' if pd.notna(v) and v > 0 else '#2ca02c' for v in d[delta_col].fillna(0.0)]
        ax.bar(x, d[delta_col], color=colors, edgecolor='k', linewidth=0.6)
        ax.axhline(0, color='k', lw=1.0)
        ax.set_ylabel(f'{metric} [decades]')
        ax.set_title('Positive = stress-supervised model lower error')
        ax.grid(axis='y', alpha=0.3)
    for ax in axes[1, :]:
        ax.set_xticks(x)
        ax.set_xticklabels(tick_labels, rotation=20, ha='right')
    fig.suptitle(f'{model_family} ({fp_status}): with-stress vs no-stress{chr(10)}Delta convention: no-stress − stress')
    fig.tight_layout()
    png, pdf = save_fig(fig, f'stress_vs_no_stress_{model_family}_{fp_status}_mae_rmse')
    figure_inventory.append({'pair_id': pair_id, 'png': str(png), 'pdf': str(pdf), 'png_exists': png.exists(), 'pdf_exists': pdf.exists()})

eh.save_json(figure_inventory, RESULTS_DIR, 'figure_inventory')
display(pd.DataFrame(figure_inventory) if figure_inventory else pd.DataFrame(columns=['pair_id', 'png', 'pdf', 'png_exists', 'pdf_exists']))


## Cautious conclusion

The final cell writes a short report answering whether auxiliary stress supervision helps fatigue-life prediction overall, in critical low-life nodes, and specifically in the lower transition.


In [ ]:

def directional_statement(values, tol=1e-3):
    vals = pd.Series(values, dtype='float64').dropna()
    if vals.empty:
        return {'label': 'insufficient evidence', 'median': np.nan, 'fraction_positive': np.nan, 'n': 0}
    median = float(vals.median())
    frac_positive = float((vals > 0).mean())
    if median > tol and frac_positive >= 0.60:
        label = 'suggests improvement from stress supervision'
    elif median < -tol and frac_positive <= 0.40:
        label = 'suggests lower error without stress supervision'
    else:
        label = 'is mixed / model-dependent'
    return {'label': label, 'median': median, 'fraction_positive': frac_positive, 'n': int(len(vals))}


overall_mae = directional_statement(
    paired_table.loc[paired_table['life_bin'] == eh.FULL_TEST_SET_LABEL, 'delta_mae_loglife'] if not paired_table.empty else []
)
overall_rmse = directional_statement(
    paired_table.loc[paired_table['life_bin'] == eh.FULL_TEST_SET_LABEL, 'delta_rmse_loglife'] if not paired_table.empty else []
)
critical_low_life = directional_statement(
    paired_table.loc[paired_table['life_bin'] == 'log_life < 2', 'delta_mae_loglife'] if not paired_table.empty else []
)

pair_lines = valid_pairs[[
    'model_family', 'fp_status', 'stress_variant', 'no_stress_variant',
    'stress_checkpoint_path', 'no_stress_checkpoint_path'
]].to_markdown(index=False) if not valid_pairs.empty else 'No valid pair discovered.'

summary_parts = [
    '# Joint stress supervision validation report',
    '',
    f'Repository commit: `{COMMIT}`',
    '',
    f'Evaluation label: **{EVALUATION_LABEL}**.',
    '',
    'Checkpoint pairing now uses verified metadata instead of the previous brittle `fair_families` rule.',
    'The prior empty-summary failure was caused by the old pairing pipeline treating missing checkpoint split metadata as `NaN` rather than missing and by relying on hard-coded family pairing assumptions.',
    '',
    '## Valid stress/no-stress model pair(s)',
    pair_lines,
    '',
    '- No nonexistent regular `PointNetMLPJoint` no-stress model was included.',
    '- Delta convention used everywhere: `delta = no_stress - stress`; positive means the stress-supervised model has lower error.',
    '',
    '## Quantitative outcome',
    f"- Full test set MAE: {overall_mae['label']} (median Δ `{overall_mae['median']:+.4f}` decades).",
    f"- Full test set RMSE: {overall_rmse['label']} (median Δ `{overall_rmse['median']:+.4f}` decades).",
    f"- Critical short-life bin `log_life < 2`: {critical_low_life['label']} (median Δ `{critical_low_life['median']:+.4f}` decades).",
    '',
]
if not dataset_available:
    summary_parts.extend([
        '## Limitation',
        f'- Dataset unavailable: `{DATASET_PATH}`. Notebook execution still completed and saved checkpoint, pairing, and empty schema-stable result artifacts, but no inference figures/metrics could be regenerated in this environment.',
        '',
    ])
if valid_pairs.empty:
    summary_parts.extend([
        '## Pairing diagnostic',
        no_pair_diagnostic,
        '',
    ])
summary = chr(10).join(summary_parts)
(RESULTS_DIR / 'validation_report.md').write_text(summary, encoding='utf-8')
eh.save_json({
    'commit': COMMIT,
    'dataset_available': bool(dataset_available),
    'valid_pairs': valid_pairs.to_dict(orient='records'),
    'overall_mae': overall_mae,
    'overall_rmse': overall_rmse,
    'critical_low_life_mae': critical_low_life,
}, RESULTS_DIR, 'validation_report')
display(Markdown(summary))
